# MindSync Stage 1 — multi-seed retraining (seeds 43, 44)

**Purpose.** Seed variance for the paper. Retrains the Stage 1 text classifier (RoBERTa-large on GoEmotions → 4 clusters) with two additional seeds, using the same data processing, model, optimizer, LR schedule, sampler and number of epochs as the original `mindsync_colab_stage1_text.ipynb`.

**Identical to the original notebook**
- GoEmotions `simplified` config (`google-research-datasets/go_emotions`), official train / validation / test splits
- 28 GoEmotions labels → 4 clusters (same dict, incl. `neutral` → Ambiguity); the **first** label of each comment is used; rows with no label are filtered out
- `roberta-large` tokenizer, `padding='max_length'`, `truncation=True`, `MAX_LEN = 128`
- `MindSyncTextModel`: CLS → Dropout → Linear 1024→512 → GELU → Dropout → Linear 512→4 (dropout 0.1)
- AdamW (lr 2e-5, weight decay 0.01), linear schedule with 10 % warmup, grad-norm clip 1.0, batch 16 (eval batch 32), 5 epochs, cross-entropy
- `WeightedRandomSampler` with inverse class-frequency weights, `replacement=True`, `num_samples = len(train)`
- Model selection: epoch with the highest **validation macro-F1**; that checkpoint is evaluated on the test split

**Intentional differences**
1. **Mixed-precision training** (`autocast` fp16 + `GradScaler`; master weights stay fp32) so that two seeds fit well inside Kaggle's 12-hour limit. The previously published run (referred to as seed 42) was trained in fp32. Validation and test inference here run in **fp32**, exactly like the original `evaluate()`. Set `USE_AMP = False` in the config cell to train in fp32 instead.
2. **Explicit seeding**: `random`, `numpy`, `torch`, `torch.cuda`, plus seeded `torch.Generator`s for the sampler and the DataLoader. (The original notebook did not set an explicit RNG seed.)
3. Headless-run housekeeping only: no checkpoints are saved, no Google Drive, no Hugging Face login/upload; `tqdm` bars are replaced by periodic `print(..., flush=True)` lines.
4. In addition to the best-validation checkpoint, the test metrics of the **last-epoch** weights are recorded (`test_last_epoch`), because the original notebook printed its test score from the model after the final epoch while uploading the best-validation checkpoint. The headline numbers are `test_best_val`.

**Output.** `/kaggle/working/stage1_multiseed_results.json` (re-written after every seed, so partial results survive a timeout) and a `SUMMARY` block at the end.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.45.2', 'datasets==2.21.0', 'huggingface_hub==0.25.2'], check=True)

import os, gc, json, time, random, platform, pathlib, traceback
from collections import Counter
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import transformers, datasets, sklearn
OUT_DIR = pathlib.Path('/kaggle/working'); OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUT_DIR / 'stage1_multiseed_results.json'
device = torch.device('cuda')
ENV = {'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
       'transformers': transformers.__version__, 'datasets': datasets.__version__,
       'sklearn': sklearn.__version__, 'gpu': torch.cuda.get_device_name(0)}
print(ENV, flush=True)

## 1 · Config

In [ ]:
SEEDS = [43, 44]
MODEL_NAME = 'roberta-large'
MAX_LEN = 128
BATCH = 16
EVAL_BATCH = 32
EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.1
CLIP_NORM = 1.0
USE_AMP = False      # the only intentional training difference vs. the original fp32 run
LOG_EVERY = 500     # progress print interval (training steps)

# Previously published Stage 1 run -- hard-coded, NOT re-run in this notebook.
PUBLISHED_RUN = {
    'seed': 42, 'test_acc': 0.7076, 'test_macro_f1': 0.6822,
    'source': 'previously published run of mindsync_colab_stage1_text.ipynb (fp32 training); '
              'hard-coded here, not re-run. Note: that notebook does not set an explicit RNG seed.',
}

# Mixed-precision helpers (torch.amp on torch>=2.3, torch.cuda.amp fallback on older torch)
try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    def amp_autocast(): return _autocast('cuda', dtype=torch.float16, enabled=USE_AMP)
    def make_scaler(): return _GradScaler('cuda', enabled=USE_AMP)
except ImportError:
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    def amp_autocast(): return _autocast(dtype=torch.float16, enabled=USE_AMP)
    def make_scaler(): return _GradScaler(enabled=USE_AMP)

## 2 · Cluster mapping (identical to the original Stage 1 notebook)

In [ ]:
EMOTION_TO_CLUSTER = {
    'grief': 'Distress', 'nervousness': 'Distress', 'fear': 'Distress',
    'sadness': 'Distress', 'remorse': 'Distress',
    'joy': 'Resilience', 'admiration': 'Resilience', 'excitement': 'Resilience',
    'relief': 'Resilience', 'amusement': 'Resilience', 'approval': 'Resilience',
    'curiosity': 'Resilience', 'desire': 'Resilience', 'gratitude': 'Resilience',
    'love': 'Resilience', 'optimism': 'Resilience', 'pride': 'Resilience',
    'realization': 'Resilience',
    'anger': 'Aggression', 'annoyance': 'Aggression', 'disgust': 'Aggression',
    'disapproval': 'Aggression', 'embarrassment': 'Aggression',
    'confusion': 'Ambiguity', 'disappointment': 'Ambiguity', 'surprise': 'Ambiguity',
    'caring': 'Ambiguity', 'neutral': 'Ambiguity',
}
CLUSTERS = ['Distress', 'Resilience', 'Aggression', 'Ambiguity']
CLUSTER_TO_IDX = {c: i for i, c in enumerate(CLUSTERS)}
assert len(EMOTION_TO_CLUSTER) == 28

## 3 · Load + preprocess GoEmotions (identical to the original)

In [ ]:
from datasets import load_dataset
ds = load_dataset('google-research-datasets/go_emotions', 'simplified')
label_names = ds['train'].features['labels'].feature.names
print('Splits:', {k: len(v) for k, v in ds.items()}, flush=True)
print('Labels:', len(label_names), flush=True)
assert set(label_names) == set(EMOTION_TO_CLUSTER), sorted(set(label_names) ^ set(EMOTION_TO_CLUSTER))

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def map_to_cluster_idx(label_ids):
    if not label_ids: return None
    return CLUSTER_TO_IDX[EMOTION_TO_CLUSTER.get(label_names[label_ids[0]], 'Ambiguity')]

def preprocess(batch):
    enc = tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LEN)
    enc['labels'] = [map_to_cluster_idx(l) for l in batch['labels']]
    return enc

ds_proc = ds.map(preprocess, batched=True, remove_columns=ds['train'].column_names)
ds_proc = ds_proc.filter(lambda x: x['labels'] is not None)
ds_proc.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
DATA_SIZES = {k: len(v) for k, v in ds_proc.items()}
CLASS_COUNTS = {k: {CLUSTERS[c]: int(n) for c, n in sorted(Counter(ds_proc[k]['labels'].tolist()).items())}
                for k in ds_proc}
print(DATA_SIZES, flush=True)
print(CLASS_COUNTS, flush=True)

## 4 · MindSyncTextModel (identical to the original)

In [ ]:
from transformers import RobertaModel

class MindSyncTextModel(nn.Module):
    HIDDEN_SIZE = 1024
    def __init__(self, model_name='roberta-large', num_classes=4, dropout=0.1):
        super().__init__()
        self.encoder = nn.Module(); self.encoder.encoder = RobertaModel.from_pretrained(model_name)
        self.encoder.hidden_size = self.encoder.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Module()
        h = self.encoder.hidden_size
        self.classifier.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(h, h // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(h // 2, num_classes),
        )
    def forward(self, input_ids, attention_mask):
        out = self.encoder.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        logits = self.classifier.classifier(cls)
        return {'embedding': cls, 'logits': logits}

## 5 · Training / evaluation helpers
Same optimizer, schedule, sampler, loss, clipping and model selection as the original; only the forward/backward pass of training runs under fp16 autocast with a `GradScaler` (gradients are unscaled before clipping).

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# weighted sampler weights for class balance (same as original)
train_labels = ds_proc['train']['labels'].tolist()
counts = Counter(train_labels)
weights = [1.0 / counts[l] for l in train_labels]
val_loader = DataLoader(ds_proc['validation'], batch_size=EVAL_BATCH)
test_loader = DataLoader(ds_proc['test'], batch_size=EVAL_BATCH)

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def predict(model, loader):
    # fp32 inference, same as the original evaluate()
    model.eval()
    preds, gold = [], []
    for batch in loader:
        ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
        out = model(ids, mask)
        preds.extend(out['logits'].argmax(-1).cpu().tolist())
        gold.extend(batch['labels'].tolist())
    return gold, preds

def test_metrics(gold, preds):
    return {'accuracy': float(accuracy_score(gold, preds)),
            'macro_f1': float(f1_score(gold, preds, average='macro')),
            'classification_report': classification_report(gold, preds, labels=list(range(4)), target_names=CLUSTERS,
                                                           digits=4, output_dict=True),
            'confusion_matrix': confusion_matrix(gold, preds, labels=list(range(4))).tolist(),
            'n': len(gold)}

def _json_default(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(f'not JSON serializable: {type(o)}')

def mean_std(xs):
    xs = np.asarray(xs, dtype=float)
    return {'n': int(len(xs)), 'values': xs.tolist(),
            'mean': float(xs.mean()) if len(xs) else None,
            'std_ddof1': float(xs.std(ddof=1)) if len(xs) > 1 else None}

def aggregate(res):
    done = [r for r in res['runs'].values() if r.get('status') == 'done']
    pub = res['published_run']
    acc_new = [r['test_best_val']['accuracy'] for r in done]
    f1_new = [r['test_best_val']['macro_f1'] for r in done]
    return {'seeds_all': [pub['seed']] + [r['seed'] for r in done],
            'accuracy_all': mean_std([pub['test_acc']] + acc_new),
            'macro_f1_all': mean_std([pub['test_macro_f1']] + f1_new),
            'seeds_new_only': [r['seed'] for r in done],
            'accuracy_new_only': mean_std(acc_new),
            'macro_f1_new_only': mean_std(f1_new),
            'note': 'Test metrics of the best-validation-macro-F1 checkpoint. std = sample std (ddof=1). '
                    'Seed 42 values are the hard-coded published run (fp32 training, no explicit seed in that notebook); '
                    'seeds 43/44 are trained in this notebook (AMP if config.use_amp).'}

def save_results():
    RESULTS['aggregate'] = aggregate(RESULTS)
    tmp = RESULTS_PATH.with_suffix('.json.tmp')
    with open(tmp, 'w') as f:
        json.dump(RESULTS, f, indent=1, default=_json_default)
    os.replace(tmp, RESULTS_PATH)

RESULTS = {
    'config': {'seeds': SEEDS, 'model_name': MODEL_NAME, 'max_len': MAX_LEN, 'batch': BATCH, 'eval_batch': EVAL_BATCH,
               'epochs': EPOCHS, 'lr': LR, 'weight_decay': WEIGHT_DECAY, 'warmup_frac': WARMUP_FRAC,
               'clip_norm': CLIP_NORM, 'use_amp': USE_AMP, 'eval_precision': 'fp32',
               'sampler': 'WeightedRandomSampler(1/class_count, replacement=True, num_samples=len(train))',
               'label_rule': 'first GoEmotions label -> cluster; empty-label rows filtered',
               'model_selection': 'best validation macro-F1 (strictly greater)'},
    'environment': ENV,
    'data': {'sizes': DATA_SIZES, 'class_counts': CLASS_COUNTS},
    'published_run': PUBLISHED_RUN,
    'runs': {},
}

In [ ]:
def run_seed(seed):
    t_start = time.time()
    set_all_seeds(seed)
    g_sampler = torch.Generator().manual_seed(seed)
    g_loader = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(weights, num_samples=len(train_labels), replacement=True, generator=g_sampler)
    train_loader = DataLoader(ds_proc['train'], batch_size=BATCH, sampler=sampler, generator=g_loader)

    model = MindSyncTextModel(model_name=MODEL_NAME, num_classes=4).to(device)
    optim = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * EPOCHS
    sched = get_linear_schedule_with_warmup(optim, int(WARMUP_FRAC * total_steps), total_steps)
    scaler = make_scaler()

    history = []
    run = {'seed': seed, 'status': 'running', 'steps_per_epoch': len(train_loader), 'total_steps': total_steps,
           'history': history}
    RESULTS['runs'][str(seed)] = run
    best_f1, best_epoch, best_state = 0.0, None, None
    skipped_total = 0

    for ep in range(1, EPOCHS + 1):
        t_ep = time.time()
        model.train(); total = 0.0; n = 0; skipped = 0
        for step, batch in enumerate(train_loader, 1):
            ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
            y = batch['labels'].to(device)
            optim.zero_grad()
            with amp_autocast():
                out = model(ids, mask)
                loss = F.cross_entropy(out['logits'].float(), y)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            scale_before = scaler.get_scale()
            scaler.step(optim); scaler.update()
            if scaler.get_scale() < scale_before: skipped += 1   # inf/nan grads -> optimizer step skipped
            sched.step()
            total += loss.item() * y.size(0); n += y.size(0)
            if step % LOG_EVERY == 0:
                el = time.time() - t_ep
                print(f'  seed {seed} | epoch {ep} | step {step}/{len(train_loader)} | loss {total/n:.4f} | '
                      f'{el/step:.3f} s/step | epoch ETA {el/step*(len(train_loader)-step)/60:.1f} min', flush=True)
        train_time = time.time() - t_ep
        skipped_total += skipped

        gold, preds = predict(model, val_loader)
        val_acc = float(accuracy_score(gold, preds)); val_f1 = float(f1_score(gold, preds, average='macro'))
        improved = val_f1 > best_f1
        if improved:
            best_f1, best_epoch = val_f1, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}   # deep copy in CPU RAM
        history.append({'epoch': ep, 'train_loss': total / n, 'val_acc': val_acc, 'val_macro_f1': val_f1,
                        'train_time_s': train_time, 'epoch_time_s': time.time() - t_ep,
                        'amp_skipped_steps': skipped, 'lr_end': sched.get_last_lr()[0]})
        print(f'Seed {seed} | Epoch {ep} | loss {total/n:.4f} | val acc {val_acc:.4f} | val F1 {val_f1:.4f} | '
              f'{(time.time()-t_ep)/60:.1f} min' + ('  <- best' if improved else ''), flush=True)

    # test: last-epoch weights (what the original notebook printed), then best-val checkpoint (headline)
    gold, preds = predict(model, test_loader)
    test_last = test_metrics(gold, preds)
    assert best_state is not None, 'validation macro-F1 never exceeded 0'
    model.load_state_dict(best_state)
    gold, preds = predict(model, test_loader)
    test_best = test_metrics(gold, preds)

    run.update({'status': 'done', 'best_epoch': best_epoch, 'best_val_macro_f1': best_f1,
                'test_best_val': test_best, 'test_last_epoch': test_last,
                'amp_skipped_steps_total': skipped_total, 'wall_time_s': time.time() - t_start})
    print(f'\nSEED {seed} FINAL TEST (best-val epoch {best_epoch}): acc {test_best["accuracy"]:.4f} | '
          f'macro-F1 {test_best["macro_f1"]:.4f}', flush=True)
    print(f'SEED {seed} last-epoch test: acc {test_last["accuracy"]:.4f} | macro-F1 {test_last["macro_f1"]:.4f}', flush=True)
    print(classification_report(gold, preds, labels=list(range(4)), target_names=CLUSTERS, digits=4), flush=True)
    print('Confusion matrix (rows = true, cols = predicted):', flush=True)
    print(np.array(test_best['confusion_matrix']), flush=True)
    print(f'SEED {seed} wall time {(time.time()-t_start)/60:.1f} min', flush=True)

    del model, optim, sched, scaler, best_state, train_loader, sampler

## 6 · Run seeds 43 and 44
Results JSON is re-written after every seed. GPU memory is released between seeds.

In [ ]:
save_results()
T0 = time.time()
for seed in SEEDS:
    print(f'\n========== SEED {seed} ==========', flush=True)
    try:
        run_seed(seed)
    except Exception as e:
        RESULTS['runs'].setdefault(str(seed), {'seed': seed})
        RESULTS['runs'][str(seed)].update({'status': 'failed', 'error': repr(e), 'traceback': traceback.format_exc()})
        print(f'SEED {seed} FAILED: {e!r}', flush=True)
        traceback.print_exc()
    finally:
        gc.collect(); torch.cuda.empty_cache()
    save_results()
    print(f'[saved {RESULTS_PATH}] total elapsed {(time.time()-T0)/3600:.2f} h | '
          f'GPU mem allocated after cleanup {torch.cuda.memory_allocated()/1e9:.2f} GB', flush=True)

## 7 · SUMMARY

In [ ]:
save_results()
agg = RESULTS['aggregate']
fmt = lambda x: 'n/a' if x is None else f'{x:.4f}'
print('===== SUMMARY START =====')
print(json.dumps(RESULTS, indent=1, default=_json_default))
print()
print('Stage 1 test metrics (best-validation-macro-F1 checkpoint, GoEmotions test split):')
print(f"  seed 42 | PUBLISHED RUN (hard-coded, fp32, not re-run here) | acc {PUBLISHED_RUN['test_acc']:.4f} | "
      f"macro-F1 {PUBLISHED_RUN['test_macro_f1']:.4f}")
for s in SEEDS:
    r = RESULTS['runs'].get(str(s), {})
    if r.get('status') == 'done':
        print(f"  seed {s} | this notebook ({'AMP' if USE_AMP else 'fp32'} training) | acc {r['test_best_val']['accuracy']:.4f} | "
              f"macro-F1 {r['test_best_val']['macro_f1']:.4f} | best epoch {r['best_epoch']} | "
              f"wall {r['wall_time_s']/60:.1f} min")
    else:
        print(f"  seed {s} | {r.get('status', 'not run').upper()} {r.get('error', '')}")
a, f = agg['accuracy_all'], agg['macro_f1_all']
print(f"  mean ± std (ddof=1) over seeds {agg['seeds_all']} (n={a['n']}): "
      f"acc {fmt(a['mean'])} ± {fmt(a['std_ddof1'])} | macro-F1 {fmt(f['mean'])} ± {fmt(f['std_ddof1'])}")
a, f = agg['accuracy_new_only'], agg['macro_f1_new_only']
print(f"  mean ± std (ddof=1) over new seeds only {agg['seeds_new_only']} (n={a['n']}): "
      f"acc {fmt(a['mean'])} ± {fmt(a['std_ddof1'])} | macro-F1 {fmt(f['mean'])} ± {fmt(f['std_ddof1'])}")
print('===== SUMMARY END =====')